# Baseline training: fine-tuning ResNet on EuroSAT

This notebook builds up the training pipeline step by step.

In [1]:
from pathlib import Path

from torchvision.datasets import EuroSAT

DATA_ROOT = Path("../data")

## Load the dataset

In [2]:
dataset = EuroSAT(root=DATA_ROOT, download=True)

print(f"Number of images: {len(dataset)}")
print(f"Classes: {dataset.classes}")

Number of images: 27000
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## Split into train / val / test (stratified)

70/15/15, stratified so each split keeps the same class proportions as the full dataset. Done in two steps: first carve out the test set, then split the remainder into train/val. The test set is set aside now and won't be touched again until final evaluation.

In [3]:
from sklearn.model_selection import train_test_split

labels = [label for _, label in dataset.samples]
indices = list(range(len(dataset)))

train_idx, test_idx = train_test_split(
    indices, test_size=0.15, stratify=labels, random_state=42,
)
train_labels = [labels[i] for i in train_idx]
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.15 / 0.85, stratify=train_labels, random_state=42,
)

print(f"Train: {len(train_idx):5d} ({len(train_idx) / len(dataset):.1%})")
print(f"Val:   {len(val_idx):5d} ({len(val_idx) / len(dataset):.1%})")
print(f"Test:  {len(test_idx):5d} ({len(test_idx) / len(dataset):.1%})")

Train: 18899 (70.0%)
Val:    4051 (15.0%)
Test:   4050 (15.0%)


### Sanity check: class balance preserved across splits

In [4]:
import numpy as np

def class_distribution(idx):
    counts = np.bincount([labels[i] for i in idx], minlength=len(dataset.classes))
    return counts / counts.sum()

print(f"{'class':25s} {'train':>8s} {'val':>8s} {'test':>8s}")
for c, class_name in enumerate(dataset.classes):
    train_pct = class_distribution(train_idx)[c]
    val_pct = class_distribution(val_idx)[c]
    test_pct = class_distribution(test_idx)[c]
    print(f"{class_name:25s} {train_pct:8.1%} {val_pct:8.1%} {test_pct:8.1%}")

class                        train      val     test
AnnualCrop                   11.1%    11.1%    11.1%
Forest                       11.1%    11.1%    11.1%
HerbaceousVegetation         11.1%    11.1%    11.1%
Highway                       9.3%     9.3%     9.3%
Industrial                    9.3%     9.3%     9.3%
Pasture                       7.4%     7.4%     7.4%
PermanentCrop                 9.3%     9.3%     9.3%
Residential                  11.1%    11.1%    11.1%
River                         9.3%     9.3%     9.3%
SeaLake                      11.1%    11.1%    11.1%


## Transforms

Normalization uses the per-channel mean/std computed in `01_explore_eurosat.ipynb` (on an EuroSAT subsample), not the ImageNet ones — we're fine-tuning, not doing frozen feature extraction, so it's worth matching the target domain. Augmentation (random flips) is applied to the train split only: satellite tiles have no natural "up", so flipping is a safe, effective augmentation. Val/test only get normalization, so evaluation stays representative of real data.

In [5]:
from torchvision import transforms

EUROSAT_MEAN = [0.3447, 0.3802, 0.4072]
EUROSAT_STD = [0.2048, 0.1396, 0.1179]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(EUROSAT_MEAN, EUROSAT_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(EUROSAT_MEAN, EUROSAT_STD),
])

## DataLoaders

`dataset` only has one transform, but train needs augmentation while val/test don't. So we create two `EuroSAT` instances pointing at the same files with different transforms, and apply the already-computed split indices to each via `Subset`.

In [6]:
from torch.utils.data import DataLoader, Subset

train_dataset_full = EuroSAT(root=DATA_ROOT, transform=train_transform)
eval_dataset_full = EuroSAT(root=DATA_ROOT, transform=eval_transform)

train_set = Subset(train_dataset_full, train_idx)
val_set = Subset(eval_dataset_full, val_idx)
test_set = Subset(eval_dataset_full, test_idx)

BATCH_SIZE = 64

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 296
Val batches:   64
Test batches:  64


### Sanity check: pull one batch

In [7]:
images, targets = next(iter(train_loader))

print(f"Batch images shape: {images.shape}")  # (batch_size, 3, 64, 64)
print(f"Batch targets shape: {targets.shape}")
print(f"Pixel value range after normalization: [{images.min():.2f}, {images.max():.2f}]")

Batch images shape: torch.Size([64, 3, 64, 64])
Batch targets shape: torch.Size([64])
Pixel value range after normalization: [-1.66, 5.03]


## Next step

Define the model: load a pre-trained ResNet, replace its final layer for 10 classes, and move it to the GPU.